In [95]:
import pandas as pd
import numpy as np 

In [96]:
hotel_data = pd.read_csv(r'../data/raw/hotel_bookings.csv')
df = hotel_data.copy()

## Dealing with the missing values

In [97]:
# from data_understanding we know the missing values
# children , country , agent , company
# Filter to show only columns with missing values
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary[missing_summary['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

,Missing Count,Percentage (%)
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


In [98]:
df[df['children'].isnull()]['children']

40600   NaN
40667   NaN
40679   NaN
41160   NaN
Name: children, dtype: float64

In [99]:
# imputing the 0 in 4 places in children col
df['children'] = df['children'].fillna(0)
df['children'].isnull().sum()

np.int64(0)

In [100]:
# missing values in country
df[df['country'].isnull()]['country']

30        NaN
4127      NaN
7092      NaN
7860      NaN
8779      NaN
         ... 
65908     NaN
65909     NaN
65910     NaN
80830     NaN
101488    NaN
Name: country, Length: 488, dtype: str

In [101]:
# fixing nan values in country
df['country'] = df['country'].fillna('Unknown')

In [102]:
# working on agent col
df[df['agent'].isnull()]['agent']

0        NaN
1        NaN
2        NaN
6        NaN
18       NaN
          ..
119124   NaN
119151   NaN
119166   NaN
119215   NaN
119248   NaN
Name: agent, Length: 16340, dtype: float64

In [127]:
# fixing nan values in agent col 
df['agent'] = df['agent'].fillna('Unknown')

In [128]:
df['agent'].isnull().sum()

np.int64(0)

In [105]:
# working with company col
df[df['company'].isnull()]['company']
# since the number of nan in company cols are very large we will we will drop the original col and replace it with has_company which will have
# value of 0 for those who don't have company(id) value and 1 for those who have company value

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
          ..
119385   NaN
119386   NaN
119387   NaN
119388   NaN
119389   NaN
Name: company, Length: 112593, dtype: float64

In [106]:
# fixing the company col
# not null gives false for the nan value and true for the values and int converts into 1 and 0
df['has_company'] = df['company'].notnull().astype('int64')
df = df.drop(columns = ['company'])

In [107]:
# checking all nan values are gone or not
print(df[['children', 'country', 'agent', 'has_company']].isnull().sum())
# df.isnull().sum()

children       0
country        0
agent          0
has_company    0
dtype: int64


## Removing whitespaces and data_inconsistency in meals column

In [108]:
# removing whitespaces from categorical columns 
cat_columns = df.select_dtypes(include=['object','string']).columns
for col in cat_columns :
    df[col] = df[col].str.strip()

# removing data inconsistency in meal
# undefined ---> sc
# since sc means self catering i.e no meal included in room booking and undefined means unknown meal therefore these two reflects same meaning 
# we will make all undefined to sc
# df['meal'].unique()
df['meal'] = df['meal'].replace('Undefined' , 'SC')
df['meal'].unique()

<ArrowStringArray>
['BB', 'FB', 'HB', 'SC']
Length: 4, dtype: str

In [109]:
## Fixing datatypes of reservation_date col from str to datetime and children col from float to int
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])
df['children'] = df['children'].astype('int64')

## Removing duplicates


In [110]:
print(df.duplicated().sum())
df = df.drop_duplicates()
df.duplicated().sum()

32011


np.int64(0)

## Dropping unnecessary columns

In [111]:
# Drop 'reservation_status' because it tells us if a guest checked out or canceled,
# which causes direct target leakage when predicting 'is_canceled'!
if 'reservation_status' in df.columns:
    df = df.drop(columns=['reservation_status'])
    print("Dropped 'reservation_status' column")

Dropped 'reservation_status' column


## Handling incorrect values in columns 

In [112]:
# in adr typo mistake of 5400 needs to be removed
if 'adr' in df.columns :
    df = df[df['adr'] < 5000]

# removing zero guest row 
if set(['adults', 'children', 'babies']).issubset(df.columns):
    zero_guest_mask = (df['adults'] + df['children'] + df['babies']) == 0
    print(zero_guest_mask.sum())
    # Keep rows where zero_guest_mask is False
    df = df[~zero_guest_mask]

166


## Verifying and uploading clean data to data\processed

In [129]:
%load_ext autoreload
%autoreload 2

import sys

# 1. Add the main project root folder to sys.path (NOT the file itself)
sys.path.append('..')

from src.hotel_booking_cancelation_prediction.cleaning import clean_data

# raw dataset
print("Raw Shape:", hotel_data.shape)

# Run full cleaning pipeline
clean_df = clean_data(hotel_data)
print("Clean Shape:", clean_df.shape)

# Save processed dataset
clean_df.to_csv('../data/processed/hotel_bookings_cleaned.csv', index=False)
print("Successfully saved clean dataset to data/processed/hotel_bookings_cleaned.csv!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Raw Shape: (119390, 32)
Clean Shape: (87212, 31)
clean agent null values: 75075
